# Cleaning data using pandas 

In [556]:
import glob

In [557]:
files = glob.glob("C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk/data/Birmingham/raw/*.csv")
print(files)

['C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk/data/Birmingham/raw\\BIRR_2021.csv', 'C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk/data/Birmingham/raw\\BIRR_2022.csv', 'C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk/data/Birmingham/raw\\BIRR_2023.csv', 'C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk/data/Birmingham/raw\\BIRR_2024.csv', 'C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk/data/Birmingham/raw\\BIRR_2025.csv', 'C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk/data/Birmingham/raw\\BIRR_weather_2021_2024.csv']


In [558]:
import pandas as pd

In [559]:
df = pd.read_csv(files[4], skiprows=4)

Change index from 0-4 to access all 5 files

In [560]:
# List the columns you want — SO2 might not exist
cols_to_get = [
    "Date",
    "time",
    "Nitrogen dioxide",
    "PM<sub>2.5</sub> particulate matter (Hourly measured)",
    "PM<sub>10</sub> particulate matter (Hourly measured)",
    "Ozone",
    "Sulphur dioxide"
]

# Only keep columns that actually exist in this file
existing_cols = [c for c in cols_to_get if c in df.columns]
df = df[existing_cols]

# If SO2 was missing, add it back as a column full of NaN
if "Sulphur dioxide" not in df.columns:
    df["Sulphur dioxide"] = float('nan')

In [561]:
df.replace([" ", "", "No data"], pd.NA, inplace=True)

In [562]:
df = df.ffill()

In [563]:
df.columns = [
    "date",
    "time",
    "NO2",
    "PM2.5",
    "PM10",
    "O3",
    "SO2"
]

In [564]:
df.to_csv("clean_BIRR_2025.csv", index=False)

change date from 2021-2025

# Merging

Change the codes of the file names

In [565]:
df1 = pd.read_csv("clean_BIRR_2021.csv")
df2 = pd.read_csv("clean_BIRR_2022.csv")
df3 = pd.read_csv("clean_BIRR_2023.csv")
df4 = pd.read_csv("clean_BIRR_2024.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'clean_BIRR_2021.csv'

In [ ]:
df = pd.concat([df1, df2, df3, df4], ignore_index=True)

In [ ]:
df.to_csv("BIRR_2021_24.csv", index=False)

Merging weather data

In [572]:
import os

In [574]:
CITIES = {
    'MY1':  'London',
    'BIRR': 'Birmingham',
    'MAN3': 'Manchester',
    'NEWC': 'Newcastle',
    'CARD': 'Cardiff',
}

BASE = "C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk/data/test"

for site_id, name in CITIES.items():
    print(f"\nMerging {name}...")

    # Load pollution file
    poll = pd.read_csv(f"{BASE}/raw/clean_{site_id}_2025.csv")

    # Drop blank rows
    poll = poll.dropna(subset=['date', 'time'])

    # Fix 24:00:00 → next day 00:00 (2025 files use seconds in time)
    poll['datetime'] = poll['date'] + ' ' + poll['time']
    poll['datetime'] = poll['datetime'].str.replace(' 24:00:00', ' 00:00', regex=False)
    poll['datetime'] = poll['datetime'].str.replace(' 24:00', ' 00:00', regex=False)
    poll['datetime'] = pd.to_datetime(poll['datetime'], format='mixed', dayfirst=True)
    poll.loc[poll['time'].str.startswith('24'), 'datetime'] += pd.Timedelta(days=1)
    poll = poll.drop(columns=['date', 'time'])
    # Load weather file
    wthr = pd.read_csv(f"{BASE}/raw/{site_id}_weather_2025.csv", skiprows=3)

    # Rename messy column names
    wthr = wthr.rename(columns={
        'time':                        'datetime',
        'temperature_2m (°C)':         'temp',
        'relative_humidity_2m (%)':    'humidity',
        'wind_speed_10m (km/h)':       'wind_speed',
        'surface_pressure (hPa)':      'pressure',
    })

    wthr['datetime'] = pd.to_datetime(wthr['datetime'])
    wthr = wthr[['datetime', 'temp', 'humidity', 'wind_speed', 'pressure']]

    # Round both to the hour
    poll['datetime'] = poll['datetime'].dt.floor('h')
    wthr['datetime'] = wthr['datetime'].dt.floor('h')

    # Merge
    merged = pd.merge(poll, wthr, on='datetime', how='left')

    merged['city'] = site_id
    merged = merged[['datetime', 'city', 'NO2', 'PM2.5', 'PM10', 'O3', 'SO2', 'temp', 'humidity', 'wind_speed', 'pressure']]

    # Save
    merged.to_csv(f"{BASE}/merged/{site_id}_merged_2025.csv", index=False)

    print(f"  Rows:            {len(merged):,}")
    print(f"  Columns:         {list(merged.columns)}")
    print(f"  Weather missing: {merged['temp'].isna().sum()} rows")


Merging London...
  Rows:            8,760
  Columns:         ['datetime', 'city', 'NO2', 'PM2.5', 'PM10', 'O3', 'SO2', 'temp', 'humidity', 'wind_speed', 'pressure']
  Weather missing: 1 rows

Merging Birmingham...
  Rows:            8,760
  Columns:         ['datetime', 'city', 'NO2', 'PM2.5', 'PM10', 'O3', 'SO2', 'temp', 'humidity', 'wind_speed', 'pressure']
  Weather missing: 1 rows

Merging Manchester...
  Rows:            8,760
  Columns:         ['datetime', 'city', 'NO2', 'PM2.5', 'PM10', 'O3', 'SO2', 'temp', 'humidity', 'wind_speed', 'pressure']
  Weather missing: 1 rows

Merging Newcastle...
  Rows:            8,760
  Columns:         ['datetime', 'city', 'NO2', 'PM2.5', 'PM10', 'O3', 'SO2', 'temp', 'humidity', 'wind_speed', 'pressure']
  Weather missing: 1 rows

Merging Cardiff...
  Rows:            8,760
  Columns:         ['datetime', 'city', 'NO2', 'PM2.5', 'PM10', 'O3', 'SO2', 'temp', 'humidity', 'wind_speed', 'pressure']
  Weather missing: 1 rows
